# Compute overlap vs magnitude (volcano plot)

**Do not run this notebook** — it takes a long time and is included for reproducibility only.
It shows how `rob{idx}_half.pkl` (loaded by `fig5.ipynb`) was generated.

For one exemplar oscillatory matrix, initial conditions are swept over a 2D grid of:
- **r**: perturbation magnitude (‖x₀‖)
- **cos**: cosine of the angle between x₀ and the reference direction u

At each grid point, 100 trials are run and the fraction of trials that return
to the reference orbit ('same orbit') is recorded. Results are saved as
`(p_hat, counts, baseline_rate)` in `rob{idx}_half.pkl`.

In [ ]:
import os
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['MKL_NUM_THREADS']      = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import sys
sys.path.append('../../..')
sys.path.append('../../../results')

from utils import W_from_J, init_cond, solve_rk4_springBox

In [ ]:
# --- Load data ---
# Download from Zenodo [DOI] and place under data/fig2/ and data/fig5/
data_root = '../../../data/fig2'
sigmas    = ['sigma02', 'sigma05', 'sigma1']

data = []
for s in sigmas:
    with open(f'{data_root}/{s}/Ws.pkl', 'rb') as f:
        data.append(pickle.load(f))

solutions = [[d for d in ds if not np.all(d.get('W') == 0)] for ds in data]
matrices0 = [d['W'] for d in solutions[0]]
matrices1 = [d['W'] for d in solutions[1]]
matrices2 = [d['W'] for d in solutions[2]]

behaviours0 = np.load(f'{data_root}/behaviours/behaviours0.npy', allow_pickle=True).tolist()
behaviours1 = np.load(f'{data_root}/behaviours/behaviours1.npy', allow_pickle=True).tolist()
behaviours2 = np.load(f'{data_root}/behaviours/behaviours2.npy', allow_pickle=True).tolist()

with open('../../../data/fig5/robustness.pkl', 'rb') as f:
    results = pickle.load(f)

In [ ]:
# --- Helper functions ---
OSC_LABELS = {'periodic', 'quasi-periodic'}

def find_first_oscillatory_index(behaviour_list, osc_labels=OSC_LABELS):
    for idx, lab in enumerate(behaviour_list):
        if lab in osc_labels:
            return idx
    return None

def build_panel1_inputs_from_ensembles(
        matrices0, matrices1, matrices2,
        behaviours0, behaviours1, behaviours2,
        a, b, c, ic_scale=0.1, verbose=True):
    Ws, init_conds, ensemble_ids, meta = [], [], [], []
    for ens_id, (mats, behs) in enumerate(zip(
            [matrices0, matrices1, matrices2],
            [behaviours0, behaviours1, behaviours2])):
        kept = skipped = 0
        for i, (J, beh_list) in enumerate(zip(mats, behs)):
            idx = find_first_oscillatory_index(beh_list)
            if idx is None:
                skipped += 1; continue
            W, _ = W_from_J(J, a, b, c)
            ics  = init_cond(J)
            x0   = ic_scale * ics[:, idx].copy()
            Ws.append(W); init_conds.append(x0)
            ensemble_ids.append(ens_id)
            meta.append(dict(ensemble_id=ens_id, matrix_index_in_ensemble=i,
                             ic_index=idx, behaviour_label=beh_list[idx]))
            kept += 1
        if verbose:
            print(f'Ensemble {ens_id}: kept={kept}, skipped={skipped}')
    return Ws, init_conds, ensemble_ids, meta

def compute_orbit_ellipsoid_and_scale(X_ref):
    T, d   = X_ref.shape
    center = X_ref.mean(axis=0)
    Xc     = X_ref - center
    cov    = np.cov(Xc, rowvar=False)
    eigvals, Q = np.linalg.eigh(cov)
    order  = np.argsort(eigvals)[::-1]
    eigvals, Q = eigvals[order], Q[:, order]
    r      = np.sqrt(np.maximum(eigvals, 0.0))
    Z      = Xc @ Q
    max_extent = np.max(np.abs(Z), axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        scale_per_axis = np.where(r > 1e-12, max_extent / r, 1.0)
    scale_base = max(np.max(scale_per_axis), 1.0)
    a_axes = r * scale_base
    S      = float(np.sqrt(np.mean(a_axes**2)))
    return center, Q, a_axes, S

def same_orbit_from_precomputed_tree(tree, X_noisy_chunk, eps,
                                     frac_thresh=0.95, max_factor=2.0):
    dists, _    = tree.query(X_noisy_chunk)
    frac_inside = float(np.mean(dists <= eps))
    max_dist    = float(np.max(dists))
    same_orbit  = (frac_inside >= frac_thresh) and (max_dist <= max_factor * eps)
    return same_orbit, frac_inside, max_dist, dists

def pick_matrix_idx(result, ensemble_id=2, S_target=1.0, eta_ic_target=1.0,
                    S_tol=0.25, eta_tol=0.25, top_k=10):
    """Find best-matching entry in results for a given ensemble, S, and eta_ic."""
    candidates = []
    for idx, r in enumerate(result):
        if r.get('ensemble_id') != ensemble_id:
            continue
        S, eta = float(r['S']), float(r['eta_ic_crit'])
        if abs(S - S_target) > S_tol or abs(eta - eta_ic_target) > eta_tol:
            continue
        candidates.append(((S-S_target)**2 + (eta-eta_ic_target)**2, idx, S, eta))
    if not candidates:
        for idx, r in enumerate(result):
            if r.get('ensemble_id') != ensemble_id: continue
            S, eta = float(r['S']), float(r['eta_ic_crit'])
            candidates.append(((S-S_target)**2 + (eta-eta_ic_target)**2, idx, S, eta))
    candidates.sort()
    best_score, best_idx, best_S, best_eta = candidates[0]
    print(f'Best: idx={best_idx}, ensemble_id={ensemble_id}, S={best_S:.4g}, eta_ic_crit={best_eta:.4g}')
    for score, idx, S, eta in candidates[:top_k]:
        print(f'  idx={idx:4d} | S={S:.4g} | eta_ic_crit={eta:.4g} | score={score:.3g}')
    return best_idx, result[best_idx], candidates

def unit(v, eps=1e-12):
    return v / (np.linalg.norm(v) + eps)

def random_unit_orthogonal_to(u_hat, rng):
    z = rng.normal(size=u_hat.shape[0])
    return unit(z - (z @ u_hat) * u_hat)

def sample_x_with_norm_and_cos_to_u(u, r, cosval, rng=None):
    rng   = np.random.default_rng() if rng is None else rng
    u_hat = unit(u); c = float(cosval)
    if abs(c) >= 1.0 - 1e-12:
        return r * np.sign(c) * u_hat
    v = random_unit_orthogonal_to(u_hat, rng)
    return r * (c * u_hat + np.sqrt(max(0.0, 1.0 - c*c)) * v)

def classify_decay_to_baseline(r, tail_len=200, thr=0.01, decrease_pct=20.0):
    T, N = r.shape
    norm_r = np.sqrt(np.sum(r**2, axis=1))
    tail_len = min(tail_len, T)
    activity_tail = r[-tail_len:, :]
    norm_tail = float(np.sum(np.sqrt(np.sum(activity_tail**2, axis=1))))
    l = tail_len
    baseline_thr = thr * np.sqrt(N) * l
    diag = {'norm_tail': norm_tail, 'baseline_thr': baseline_thr}
    if norm_tail < baseline_thr:
        return True, '0_fp', diag
    third = len(norm_r) // 3
    if third < 2:
        return False, 'active', diag
    first_tri_avg  = float(np.mean(norm_r[:2*third]))
    second_tri_avg = float(np.mean(norm_r[2*third:]))
    pct = (first_tri_avg - second_tri_avg) / (first_tri_avg + 1e-12) * 100.0
    diag.update({'first_2third_avg_norm': first_tri_avg,
                 'last_1third_avg_norm': second_tri_avg,
                 'percentage_decrease': pct})
    return (pct > decrease_pct), '0_fp' if pct > decrease_pct else 'active', diag

def make_orbit_reference_chunk(r_ref, burnin=0, chunk_len=2000):
    i0 = int(burnin)
    i1 = min(i0 + int(chunk_len), r_ref.shape[0])
    return r_ref[i0:i1, :]

def precompute_orbit_tree_and_eps(X_ref_chunk, alpha=0.1):
    center, Q, a, S = compute_orbit_ellipsoid_and_scale(X_ref_chunk)
    return cKDTree(X_ref_chunk), float(alpha * S), S, (center, Q, a)

def build_jobs(radii, cosines, n_reps=50, seed=0):
    jobs = []
    base_rng = np.random.default_rng(seed)
    for r in radii:
        for c in cosines:
            reps = 1 if abs(c) >= 1.0 - 1e-12 else n_reps
            rep_seeds = base_rng.integers(0, 2**31 - 1, size=reps, dtype=np.int64)
            for k, s in enumerate(rep_seeds):
                jobs.append({'r': float(r), 'c': float(c), 'rep': int(k), 'seed': int(s)})
    return jobs

def run_one_trial(job, u_dir, W, tauE, tauI, T_total, dt, a, b, c, bias,
                  tree, eps, frac_thresh=0.95, max_factor=2.0,
                  noisy_burnin=0, noisy_chunk_len=2000,
                  decay_tail_len=200, decay_thr=0.01, decay_decrease_pct=20.0):
    rng    = np.random.default_rng(job['seed'])
    x0     = sample_x_with_norm_and_cos_to_u(u_dir, job['r'], job['c'], rng=rng)
    tvec, r_noisy = solve_rk4_springBox(x0, W, tauE, tauI, T_total, dt, a, b, c, bias)
    is_baseline, _, _ = classify_decay_to_baseline(
        r_noisy, tail_len=decay_tail_len, thr=decay_thr, decrease_pct=decay_decrease_pct)
    if is_baseline:
        return {'r': job['r'], 'c': job['c'], 'rep': job['rep'],
                'is_baseline': True, 'same_orbit': False,
                'frac_inside': np.nan, 'max_dist': np.nan}
    i0 = int(noisy_burnin)
    i1 = min(i0 + int(noisy_chunk_len), r_noisy.shape[0])
    X_noisy_chunk = r_noisy[i0:i1, :]
    if X_noisy_chunk is None or len(X_noisy_chunk) < 10:
        return {'r': job['r'], 'c': job['c'], 'rep': job['rep'],
                'is_baseline': False, 'same_orbit': False,
                'frac_inside': np.nan, 'max_dist': np.nan}
    same, frac, max_d, _ = same_orbit_from_precomputed_tree(
        tree, X_noisy_chunk, eps, frac_thresh=frac_thresh, max_factor=max_factor)
    return {'r': job['r'], 'c': job['c'], 'rep': job['rep'],
            'is_baseline': False, 'same_orbit': bool(same),
            'frac_inside': float(frac), 'max_dist': float(max_d)}

def run_grid_parallel(jobs, u_dir, W, tauE, tauI, T_total, dt, a, b, c, bias,
                      tree, eps, n_jobs=-1, backend='loky', verbose=10, **kwargs):
    return Parallel(n_jobs=n_jobs, backend=backend, verbose=verbose)(
        delayed(run_one_trial)(
            job, u_dir, W, tauE, tauI, T_total, dt, a, b, c, bias, tree, eps, **kwargs)
        for job in jobs)

def aggregate_success(results, radii, cosines):
    radii   = np.array(radii,   dtype=float)
    cosines = np.array(cosines, dtype=float)
    Nr, Nc  = len(radii), len(cosines)
    idx_r = {float(r): i for i, r in enumerate(radii)}
    idx_c = {float(c): j for j, c in enumerate(cosines)}
    counts = np.zeros((Nr, Nc), dtype=int)
    succ   = np.zeros((Nr, Nc), dtype=int)
    base   = np.zeros((Nr, Nc), dtype=int)
    for rr in results:
        i = idx_r[float(rr['r'])]; j = idx_c[float(rr['c'])]
        counts[i, j] += 1
        if rr['is_baseline']:       base[i, j] += 1
        elif rr['same_orbit']:      succ[i, j] += 1
    p_hat         = np.divide(succ, counts, out=np.zeros_like(succ, dtype=float), where=counts>0)
    baseline_rate = np.divide(base, counts, out=np.zeros_like(base, dtype=float),  where=counts>0)
    return p_hat, counts, baseline_rate

In [ ]:
# --- Simulation parameters ---
N    = 100
tauE = 1
tauI = 1
T_total = 150
dt   = 0.1
bias = np.zeros(N)
a    = np.ones(N)
c    = np.ones(N)
b    = np.zeros(N)

Ws, init_conds, ensemble_ids, meta = build_panel1_inputs_from_ensembles(
    matrices0, matrices1, matrices2,
    behaviours0, behaviours1, behaviours2,
    a=a, b=b, c=c, ic_scale=0.1, verbose=True
)

In [ ]:
# --- Select exemplar matrix ---
# Use pick_matrix_idx to find a good candidate, then fix by index for reproducibility
# best_idx, best_entry, _ = pick_matrix_idx(results, ensemble_id=2, S_target=1.0, eta_ic_target=1.0)
best_idx = 1144
W     = Ws[best_idx]
u_dir = init_conds[best_idx]

# Reference trajectory
tvec, r_ref = solve_rk4_springBox(u_dir, W, tauE, tauI, T_total, dt, a, b, c, bias)

# Reference chunk + KDTree + tube radius
X_ref_chunk = make_orbit_reference_chunk(r_ref, burnin=500, chunk_len=1000)
tree, eps, S_chunk, _ = precompute_orbit_tree_and_eps(X_ref_chunk, alpha=0.1)

print(f'S_chunk={S_chunk:.4g}, eps={eps:.4g}')

In [ ]:
# --- Define grid and run ---
# WARNING: takes a long time
radii   = [0.01, 0.025, 0.05] + list(np.arange(0.1, 2.0 + 1e-12, 0.1))
cosines = list(np.round(np.arange(0, 1.0 + 1e-12, 0.05), 10))

jobs = build_jobs(radii, cosines, n_reps=100, seed=0)

trial_results = run_grid_parallel(
    jobs, u_dir, W, tauE, tauI, T_total, dt, a, b, c, bias,
    tree, eps, n_jobs=-1, backend='loky', verbose=10,
    frac_thresh=0.95, max_factor=2.0,
    noisy_burnin=500, noisy_chunk_len=1000,
    decay_tail_len=200, decay_thr=0.01, decay_decrease_pct=20.0
)

p_hat, counts, baseline_rate = aggregate_success(trial_results, radii, cosines)

In [ ]:
# --- Save ---
with open(f'../../../data/fig5/rob{best_idx}_half.pkl', 'wb') as f:
    pickle.dump((p_hat, counts, baseline_rate), f)
print('Saved.')